In [1]:
with open("the-verdict.txt", "r") as file:
    content = file.read()
    print(content[:100])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [2]:
import re 

In [3]:
class SimpleTokenizerV2:

    # this is the init method of the class and which is called each time we create an instance of the class
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    # this is the encode method of the class which takes the input text and returns a list of words integers 
    def encode(self, text):
        # what this line does is splits the texts input in list of words and punctuation based on the regex pattern provided and also keeps the punctuation in the list
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        # what this line does is removes the leading and trailing white spaces from each item in the list and also removes any empty items from the list 
        preprocessed = [
        item.strip() for item in preprocessed if item.strip()
        ]
        # what this lines does is replaces any items in the processed list that are not in vocab with the token "<|unk|>" and then converts each item in the list to its corresponding integer id using the str_to_int dictionary
        preprocessed = [item if item in self.str_to_int           
                else "<|unk|>" for item in preprocessed]
        
        #  this line converts each item in the preprocessed list to its corresponding integer id using the str_to_int dictionary and returns the list of integer ids
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    

    def decode(self, ids):
        # this line converts each item in the ids list to its corresponding string using the int_to_str dictionary and returns the list of strings
        text = " ".join([self.int_to_str[i] for i in ids])

        # this is very important line as it removes the extra spaces before the puncuation marks and also removes the extra spaces between the words and punctuation marks
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)   
        return text
    

In [4]:
all_words = sorted(list(set(content.split())))
all_words.extend(["<|unk|>" , "<|pad|>"])


# now changing the vocal to the new index
vocab = {token: i for i, token in enumerate(all_words)}

for i , token in enumerate(list(vocab.items())[-5:]):
    print(i , token)

0 ('younger', 1482)
1 ('your', 1483)
2 ('yourself', 1484)
3 ('<|unk|>', 1485)
4 ('<|pad|>', 1486)


In [5]:
tokenizer = SimpleTokenizerV2(vocab)

In [6]:
enc_text = tokenizer.encode(content)
print(len(enc_text))

4690


In [9]:
enc_sample = enc_text[:50]
print(enc_sample)

[111, 104, 254, 1314, 117, 91, 1097, 209, 378, 1485, 1485, 1313, 209, 651, 580, 530, 1485, 1203, 773, 1407, 946, 659, 1260, 1331, 880, 1331, 694, 1287, 1485, 752, 1290, 698, 963, 723, 645, 1485, 689, 665, 507, 723, 1007, 1485, 878, 209, 1120, 1485, 1485, 261, 537, 719]


In [11]:
import torch 

In [12]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

<div class="alert alert-block alert-warning">

The GPTDatasetV1 class in listing 2.5 is based on the PyTorch Dataset class.

It defines how individual rows are fetched from the dataset. 

Each row consists of a number of
token IDs (based on a max_length) assigned to an input_chunk tensor. 

The target_chunk
tensor contains the corresponding targets. 

I recommend reading on to see how the data
returned from this dataset looks like when we combine the dataset with a PyTorch
DataLoader -- this will bring additional intuition and clarity.
    
</div>

<div class="alert alert-block alert-info">
    
Step 1: Initialize the tokenizer

Step 2: Create dataset

Step 3: drop_last=True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes
during training

Step 4: The number of CPU processes to use for preprocessing
    
</div>

In [14]:
import tiktoken

In [18]:
def create_dataloader_v1(txt , batch_size , max_length = 256 , stride = 128 , shuffle = True , drop_last = True , num_workers = 0) :
    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    
    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

<div class="alert alert-block alert-success">
    
Let's test the dataloader with a batch size of 1 for an LLM with a context size of 4, 

This will develop an intuition of how the GPTDatasetV1 class and the
create_dataloader_v1 function work together: </div>

In [16]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [19]:
import torch
print("PyTorch version:", torch.__version__)
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

PyTorch version: 2.11.0+cpu
[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


<div class="alert alert-block alert-warning">

The first_batch variable contains two tensors: the first tensor stores the input token IDs,
and the second tensor stores the target token IDs. 

Since the max_length is set to 4, each of the two tensors contains 4 token IDs. 

Note that an input size of 4 is relatively small and only chosen for illustration purposes. It is common to train LLMs with input sizes of at least
256.
    
</div>

In [20]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [21]:
# this above illstrates the meaningof stride 


<div class="alert alert-block alert-warning">

If we compare the first with the second batch, we can see that the second batch's token
IDs are shifted by one position compared to the first batch. 

For example, the second ID in
the first batch's input is 367, which is the first ID of the second batch's input. 

The stride
setting dictates the number of positions the inputs shift across batches, emulating a sliding
window approach
    
</div>

In [22]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
